In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install Augmentor

In [ ]:
import Augmentor

# Specify the path to your original image dataset
original_dataset_path = "/content/drive/MyDrive/fish images"

# Specify the path to store the augmented images
output_path = "augmented"

# Create an Augmentor Pipeline
pipeline = Augmentor.Pipeline(original_dataset_path, output_directory=output_path)

# Define the augmentations you want to apply
pipeline.rotate(probability=0.7, max_left_rotation=10, max_right_rotation=10)
pipeline.flip_left_right(probability=0.5)
pipeline.flip_top_bottom(probability=0.5)
pipeline.zoom_random(probability=0.5, percentage_area=0.8)
pipeline.flip_left_right(probability=0.5)

# Specify the number of augmented images you want to generate
num_augmented_images = 1000

# Execute the augmentation process
pipeline.sample(num_augmented_images)


In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import os
import matplotlib.pyplot as plt
# Define data directory
data_dir = '/content/drive/MyDrive/fish images'


# Create an ImageDataGenerator for data augmentation
datagen = tf.keras.preprocessing.image.ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    validation_split=0.2
)

# Use image_dataset_from_directory to create a dataset
full_dataset = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    labels='inferred',
    label_mode='categorical',
    image_size=(224, 224),
    batch_size=32,
    shuffle=True,
    seed=42,
    validation_split=0.2,
    subset='training'
)

# Create a separate validation dataset
validation_dataset = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    labels='inferred',
    label_mode='categorical',
    image_size=(224, 224),
    batch_size=32,
    shuffle=True,
    seed=42,
    validation_split=0.2,
    subset='validation'
)

# Display some images from the training set before augmentation
sample_images, _ = next(iter(full_dataset))
num_original_train_images = sample_images.shape[0]
plt.figure(figsize=(12, 6))
for i in range(6):
    plt.subplot(2, 3, i + 1)
    plt.imshow(sample_images[i].numpy().astype("uint8"))
    plt.title(f"Original Train Image {i+1}")
    plt.axis('off')
plt.show()

# Display some images from the validation set before augmentation
sample_images_val, _ = next(iter(validation_dataset))
num_original_val_images = sample_images_val.shape[0]
plt.figure(figsize=(12, 6))
for i in range(6):
    plt.subplot(2, 3, i + 1)
    plt.imshow(sample_images_val[i].numpy().astype("uint8"))
    plt.title(f"Original Validation Image {i+1}")
    plt.axis('off')
plt.show()

# Use the same generator to create augmented images for the training set
augmented_images_train, _ = next(iter(full_dataset))
num_augmented_train_images = augmented_images_train.shape[0]

# Display some images from the training set after augmentation
plt.figure(figsize=(12, 6))
for i in range(6):
    plt.subplot(2, 3, i + 1)
    plt.imshow(augmented_images_train[i].numpy().astype("uint8"))
    plt.title(f"Augmented Train Image {i+1}")
    plt.axis('off')
plt.show()

# Print the number of original and augmented images for both training and validation sets
print(f"Number of original training images: {num_original_train_images}")
print(f"Number of augmented training images: {num_augmented_train_images}")
print(f"Number of original validation images: {num_original_val_images}")


In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense

# Define the CNN model
model = Sequential()
model.add(Conv2D(32, (3, 3), activation='relu', input_shape=(224, 224, 3)))
model.add(MaxPooling2D((2, 2)))
model.add(Conv2D(64, (3, 3), activation='relu'))
model.add(MaxPooling2D((2, 2)))
model.add(Conv2D(128, (3, 3), activation='relu'))
model.add(MaxPooling2D((2, 2)))
model.add(Flatten())
model.add(Dense(128, activation='relu'))
model.add(Dense(10, activation='softmax'))  # Assuming you have 10 classes

# Compile the model
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Train the model on the augmented data
history = model.fit(
    train_generator,
    steps_per_epoch=train_generator.samples // train_generator.batch_size,
    epochs=10,  # You can adjust the number of epochs
    validation_data=test_generator,
    validation_steps=test_generator.samples // test_generator.batch_size
)

# Plot the training and validation accuracy over epochs
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.show()
